In [ ]:
session_id = 2025271200001

# ✨ Microsservice 1
(Create) **CURRICULUM_VITAE** Database:  ProfessionalProfile object (cv_obj)  
(Decision) **JOBS** Database: Check on whether the set of jobs for the cv matching position needs update

In [ ]:
DB2_JOBS = []
DB1_CURRICULUM_VITAE = []
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_TEMPERATURE = 0.2

## Imports and Install libs

In [ ]:
! pip install -U pymupdf4llm[ocr,layout]
! pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.0 MB/s eta 0:00:00


In [ ]:
import os
import random
import pymupdf4llm
import pathlib
import hashlib
import json
from typing import List, Optional
from pydantic import BaseModel, Field
from datetime import timedelta, datetime

from google import genai
from google.genai import types
from google.colab import userdata

# Langchain imports for structured output
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model # Added for generic model initialization

from google.colab import drive
drive.mount('/content/drive')

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

## Define object structures

In [ ]:
class HardSkill(BaseModel):
    description: str = Field(description="Description of the hard skill.")
    time_experience: Optional[float] = Field(description="Optional time experience in months identified for the hard skill application.")

class Position(BaseModel):
    name: str = Field(description="Name of the goal position or experienced position")
    industry: List[str] = Field(description="Matching industries for the position")
    time_experience: Optional[float] = Field(description="Optional time experience in months identified for the held position.")

class ProfessionalProfile(BaseModel):
    positions: List[Position] = Field(description="Maximum of 3 names for matching positions identified in the profile description")
    main_position: List[Position] = Field(description="Main position goal identified for profile description")
    main_position_name_variations: list = Field(description="Main position name variations. Example: Senior Accountant, Accountant III, Accountant Specialist, Experienced Accountant")
    hard_skills: List[HardSkill]
    soft_skills: List[str] = Field(description="Description of the soft skill")

## Main Functions

In [ ]:
# Generate an ID for the doc
def git_hash_object(data: bytes, obj_type: str = 'blob') -> str:
    """Calculates the git hash for given data."""
    content = f"{obj_type} {len(data)}\0".encode() + data
    return hashlib.sha1(content).hexdigest()

def generate_file_doc_id(file_path: str) -> str:
    """Generates an ID for a file document."""
    with open(file_path, 'rb') as f:
        file_content_bytes = f.read()
        file_hash = git_hash_object(file_content_bytes)
    return datetime.now().strftime('%Y%m%d%H%M%S') + file_hash # Uncommented to match kernel state behavior

def extract_professional_structured_data(text: str) -> dict:
    llm = init_chat_model(
        model=LLM_MODEL_NAME,
        model_provider=LLM_PROVIDER,
        temperature=LLM_TEMPERATURE,
        api_key=userdata.get('API_KEY')
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Your role is to extract relevant data for building a professional role description object out of either a curriculum vitae text or a description for a job position out of input text"),
        ("human", "{input}")
    ])

    structured_llm = llm.with_structured_output(schema=ProfessionalProfile)
    chain = prompt | structured_llm
    response = chain.invoke({"input": text})
    return response.model_dump()

def store_data_to_db(obj: dict, db: list, **kwargs):
    new_record = {}
    for key, value in kwargs.items():
          new_record[key] = value

    new_record["json_data"] = obj
    new_record["creation_timestamp"] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    db.append(new_record)
    return new_record

In [ ]:
structured_jobs = []
JOBS_DB = [
    {
      "id": 889977,
      "created": "2025-03-01 10:30:00",
      "last_updated": "2025-04-10 16:00:00",
      "time_posted": "1mo",
      "title": "Data Engineer",
      "description": " We are seeking a Senior Data Engineer, Engineering & Operations to lead the Engineering & Automation pillar. Reporting to the Sr. Director of Engineering & Operations, you will lead the engineering and operational excellence behind our data collaboration ecosystem. You will be instrumental in designing and implementing scalable data architectures and driving automation through AI agents and self-service tooling, translating prototypes into robust production solutions. Duties and Responsibilities:   The person in this role will:   Architecture Leadership: Define partner onboarding and clean room architecture patterns across Snowflake, LiveRamp, and Databricks that are secure, scalable, and repeatable.    Infrastructure Setup & Library Deployment: Configure and manage partner-specific clean room environments; deploy and manage Python-based libraries within the platform ecosystem.    MLOps Integration: Establish and maintain MLOps practices, including model serving, monitoring, and pipeline orchestration for AI/ML features deployed within the platform ecosystem.    Security & RBAC: Own design and enforcement of granular RBAC policies and least-privilege service accounts.    Partner Onboarding: Serve as the technical lead for onboarding new partners, implementing privacy-preserving controls (e.g., aggregation thresholds and anonymization techniques).    Pipeline & Data Product Ownership: Design, build, and operate scalable ELT pipelines using Snowpark and/or PySpark and advanced SQL to provision Gold datasets.    Identity Resolution: Implement and evolve identity resolution logic mapping internal data to 3P identifiers (including LUIDs, RampIDs, TransUnion IDs), ensuring privacy-safe practices.    Scalable Architecture: Design and operate scalable data architectures across Snowflake and Databricks supporting batch and near real-time processing patterns.    Data Quality by Design: Build robust automated checks (e.g., Great Expectations or custom SQL assertions) and define quality standards to detect schema drift, null rate spikes, and volume anomalies.    FinOps & Operational Excellence: Lead performance optimization across platforms (query tuning, caching, incremental processing) and define and implement query tagging and chargeback models for accurate cost attribution.    Operational Maturity: Establish monitoring, alerting, runbooks, and standard operating procedures to improve platform reliability and reduce incident time-to-resolution.    Testing & Validation: Validate that output data adheres to privacy and business requirements, and define test strategies for partner-facing releases.    Technical Escalation: Serve as the escalation point for diagnosing connection failures, data discrepancies, or latency issues with partner technical teams.    Agentic Workflows & Platform Enablement: Design and build internal AI agents (using frameworks like LangChain, Snowflake Cortex) and mentor other engineers through code reviews, design discussions, and operational best practices.         Qualifications: Education: Bachelor’s degree or higher in Computer Science, Information Systems, Software, Electrical or Electronics Engineering. Technical Experience: 5+ years of Data Engineering experience, with deep proficiency in advanced SQL and Python. 3+ years of hands-on experience with cloud data platforms, specifically Snowflake or Databricks. Proven experience building and operating scalable ELT pipelines using orchestration tools (e.g., Airflow, dbt). Strong track record designing production-grade systems (observability, reliability, performance tuning, incident response). Specialized Skills: Clean Room Knowledge: Exposure to Data Clean Room concepts and Clean Room platforms like LiveRamp, Snowflake or Databricks. AI/LLM Experience: Experience building applications with LLMs, RAG, Vector Databases, or frameworks like LangChain/LlamaIndex. Leadership: Ability to mentor other engineers through code reviews, design discussions, and operational best practices. Certifications Preferred: SnowPro Core Certification OR Databricks Certified Data Engineer Associate. Certifications Highly Preferred: SnowPro Advanced: Data Engineer OR Databricks Certified Data Engineer Professional. Additional Requirements:     Fully Remote: This position has been designated as fully remote, meaning that the position is expected to contribute from a non-NBCUniversal worksite, most commonly an employee’s residence.      This position is eligible for company sponsored benefits, including medical, dental and vision insurance, 401(k), paid leave, tuition reimbursement, and a variety of other discounts and perks. Learn more about the benefits offered by NBC Universal by visiting the Benefits page of the Careers website. Salary range: $140,000 - $180,000 (bonus eligible)",
      "seniority": "Mid-Senior level",
      "employment_type": "Full-time",
      "location": "San Francisco, CA",
      "url": "https://www.professional-network.com/jobs/view/job-url",
      "company_id": 102938,
      "company_name": "NBC Universal",
      "company_url": "https://www.professional-network.com/company/nbc-company",
      "deleted": 0,
      "application_active": 1,
      "salary": "$20,000 - $40,000",
      "applicants_count": "23",
      "country": "United States"
    },
    {
      "id": 901122,
      "created": "2026-01-15 09:00:00",
      "last_updated": "2026-02-01 14:20:00",
      "time_posted": "2w",
      "title": "Principal Data Engineer (FinTech)",
      "description": "We are looking for a Principal Data Engineer to architect our next-generation financial streaming platform. You will lead the migration from legacy batch processing to a real-time Kafka-based architecture.\n\nKey Responsibilities:\n- Design low-latency streaming pipelines using Flink and Kafka.\n- Lead the data governance initiative across the engineering org.\n- Optimize Snowflake storage costs and query performance.\n\nQualifications:\n- 8+ years in Data Engineering.\n- Expert-level Python and Java/Scala.\n- Deep experience with AWS (Kinesis, MSK, Lambda).",
      "seniority": "Director",
      "employment_type": "Full-time",
      "location": "New York, NY",
      "url": "https://www.professional-network.com/jobs/view/901122",
      "company_id": 445566,
      "company_name": "Apex Finance Systems",
      "company_url": "https://www.professional-network.com/company/apex-finance",
      "deleted": 0,
      "application_active": 1,
      "salary": "$210,000 - $260,000",
      "applicants_count": "45",
      "country": "United States"
    },
    {
      "id": 901123,
      "created": "2026-01-20 08:30:00",
      "last_updated": "2026-01-20 08:30:00",
      "time_posted": "1w",
      "title": "Junior Data Engineer",
      "description": "Join our growing Analytics team to help maintain our dbt projects and BigQuery data warehouse. This is a great role for someone transitioning from Data Analysis into Engineering.\n\nDuties:\n- Write and optimize SQL models in dbt.\n- Monitor Airflow DAGs for failures.\n- Ensure data quality in our Looker dashboards.\n\nQualifications:\n- 1-2 years experience with SQL.\n- Basic understanding of Python and Git.\n- Knowledge of GCP is a plus.",
      "seniority": "Entry level",
      "employment_type": "Full-time",
      "location": "Austin, TX",
      "url": "https://www.professional-network.com/jobs/view/901123",
      "company_id": 778899,
      "company_name": "RetailFlow Inc.",
      "company_url": "https://www.professional-network.com/company/retailflow",
      "deleted": 0,
      "application_active": 1,
      "salary": "$85,000 - $110,000",
      "applicants_count": "156",
      "country": "United States"
    },
    {
      "id": 901124,
      "created": "2026-01-28 11:15:00",
      "last_updated": "2026-02-02 10:00:00",
      "time_posted": "5d",
      "title": "Senior Data Engineer - MLOps",
      "description": "Help us bridge the gap between Data Science and Production. You will build the pipelines that feed our LLM training sets.\n\nResponsibilities:\n- Build vector database pipelines (Pinecone/Weaviate).\n- Deploy model monitoring services.\n- Maintain high-scale PySpark jobs on Databricks.\n\nRequirements:\n- 5+ years experience.\n- Strong background in Spark and Kubernetes.\n- Previous experience with MLflow.",
      "seniority": "Mid-Senior level",
      "employment_type": "Full-time",
      "location": "Remote",
      "url": "https://www.professional-network.com/jobs/view/901124",
      "company_id": 121314,
      "company_name": "NeuralPath AI",
      "company_url": "https://www.professional-network.com/company/neuralpath",
      "deleted": 0,
      "application_active": 1,
      "salary": "$175,000 - $220,000",
      "applicants_count": "89",
      "country": "United States"
    },
    {
      "id": 901125,
      "created": "2026-01-10 16:45:00",
      "last_updated": "2026-01-30 09:00:00",
      "time_posted": "3w",
      "title": "Data Engineer (Contract)",
      "description": "6-month contract to assist with a legacy migration from On-prem Hadoop to Azure Data Factory and Synapse.\n\nTasks:\n- Refactor MapReduce jobs into Spark/Scala.\n- Map legacy schemas to new cloud formats.\n- Technical documentation.\n\nRequired:\n- Strong Azure background.\n- 4+ years of Hadoop/Hive experience.",
      "seniority": "Mid-Senior level",
      "employment_type": "Contract",
      "location": "Chicago, IL",
      "url": "https://www.professional-network.com/jobs/view/901125",
      "company_id": 554433,
      "company_name": "Global Logistics Corp",
      "company_url": "https://www.professional-network.com/company/globallogistics",
      "deleted": 0,
      "application_active": 1,
      "salary": "$80 - $100 /hr",
      "applicants_count": "12",
      "country": "United States"
    },
    {
      "id": 901126,
      "created": "2026-02-01 13:00:00",
      "last_updated": "2026-02-01 13:00:00",
      "time_posted": "1d",
      "title": "Lead Data Infrastructure Engineer",
      "description": "We are seeking a Lead Engineer to manage our core data infrastructure. You won't just build pipelines; you'll build the platform the pipelines run on.\n\nDuties:\n- Manage Terraform scripts for AWS data resources.\n- Optimize Trino/Presto clusters.\n- Implement CI/CD for data deployments.\n\nQualifications:\n- Proficiency in Go or Python.\n- Infrastructure as Code (Terraform/CloudFormation) expert.",
      "seniority": "Mid-Senior level",
      "employment_type": "Full-time",
      "location": "Seattle, WA",
      "url": "https://www.professional-network.com/jobs/view/901126",
      "company_id": 667788,
      "company_name": "CloudScale Systems",
      "company_url": "https://www.professional-network.com/company/cloudscale",
      "deleted": 0,
      "application_active": 1,
      "salary": "$190,000 - $230,000",
      "applicants_count": "31",
      "country": "United States"
    },
    {
      "id": 901127,
      "created": "2026-01-05 10:00:00",
      "last_updated": "2026-02-01 17:30:00",
      "time_posted": "1mo",
      "title": "Healthcare Data Engineer",
      "description": "Focus on building HIPAA-compliant data lakes for patient analytics. You will work closely with medical researchers.\n\nSkills:\n- FHIR/HL7 data standards experience.\n- AWS Glue and Athena.\n- Strong Python and data encryption knowledge.\n\nEducation:\n- MS in Health Informatics or CS preferred.",
      "seniority": "Mid-Senior level",
      "employment_type": "Full-time",
      "location": "Boston, MA",
      "url": "https://www.professional-network.com/jobs/view/901127",
      "company_id": 223344,
      "company_name": "BioHealth Data",
      "company_url": "https://www.professional-network.com/company/biohealth",
      "deleted": 0,
      "application_active": 1,
      "salary": "$135,000 - $170,000",
      "applicants_count": "54",
      "country": "United States"
    },
    {
      "id": 901128,
      "created": "2026-01-25 09:20:00",
      "last_updated": "2026-01-25 09:20:00",
      "time_posted": "1w",
      "title": "Data Engineer - Analytics Engineering",
      "description": "Positioned between software engineering and data analysis, you will focus on building clean, tested datasets for our business users.\n\nTech Stack:\n- dbt Cloud, Snowflake, Fivetran, and Tableau.\n- Heavy emphasis on software best practices (unit testing, documentation).",
      "seniority": "Associate",
      "employment_type": "Full-time",
      "location": "Denver, CO",
      "url": "https://www.professional-network.com/jobs/view/901128",
      "company_id": 990011,
      "company_name": "Mountain Metrics",
      "company_url": "https://www.professional-network.com/company/mountainmetrics",
      "deleted": 0,
      "application_active": 1,
      "salary": "$115,000 - $145,000",
      "applicants_count": "72",
      "country": "United States"
    },
    {
      "id": 901129,
      "created": "2026-01-12 14:00:00",
      "last_updated": "2026-01-20 12:00:00",
      "time_posted": "3w",
      "title": "ETL Developer / Data Engineer",
      "description": "Support our marketing department by consolidating data from 20+ different ad platforms into our centralized warehouse.\n\nRequired:\n- Advanced SQL skills.\n- Experience with Python API integrations.\n- Previous experience with Airbyte or Meltano is a plus.",
      "seniority": "Mid-Senior level",
      "employment_type": "Full-time",
      "location": "Miami, FL",
      "url": "https://www.professional-network.com/jobs/view/901129",
      "company_id": 332211,
      "company_name": "AdReach Agency",
      "company_url": "https://www.professional-network.com/company/adreach",
      "deleted": 0,
      "application_active": 1,
      "salary": "$120,000 - $155,000",
      "applicants_count": "41",
      "country": "United States"
    },
    {
      "id": 901130,
      "created": "2026-02-02 08:00:00",
      "last_updated": "2026-02-02 08:00:00",
      "time_posted": "1h",
      "title": "Staff Data Engineer (Data Mesh focus)",
      "description": "Help us decentralize our data architecture. You will define the standards for 'Data as a Product' across the company.\n\nExperience:\n- Implementing Data Mesh or Data Fabric concepts.\n- High-level architectural design.\n- Strong stakeholder management.",
      "seniority": "Staff",
      "employment_type": "Full-time",
      "location": "Remote",
      "url": "https://www.professional-network.com/jobs/view/901130",
      "company_id": 556677,
      "company_name": "Enterprise Scale Co",
      "company_url": "https://www.professional-network.com/company/enterprisescale",
      "deleted": 0,
      "application_active": 1,
      "salary": "$230,000 - $280,000",
      "applicants_count": "5",
      "country": "United States"
    },
    {
      "id": 901131,
      "created": "2026-01-22 11:30:00",
      "last_updated": "2026-02-01 10:00:00",
      "time_posted": "1w",
      "title": "Data Engineer - Game Analytics",
      "description": "Build the pipelines that track millions of events per second for our flagship mobile titles. \n\nSkills:\n- Java/Kotlin or Scala.\n- Google Cloud Dataflow / Apache Beam.\n- BigQuery and Looker.",
      "seniority": "Mid-Senior level",
      "employment_type": "Full-time",
      "location": "Los Angeles, CA",
      "url": "https://www.professional-network.com/jobs/view/901131",
      "company_id": 884422,
      "company_name": "Starlight Gaming",
      "company_url": "https://www.professional-network.com/company/starlight",
      "deleted": 0,
      "application_active": 1,
      "salary": "$150,000 - $190,000",
      "applicants_count": "112",
      "country": "United States"
    }
  ]

for job in JOBS_DB:
  structured_jobs.append(extract_professional_structured_data(job["description"]))

In [ ]:
structured_jobs

# 🧪 Testing Microsservice 1

## Complete Flow - Single CV

In [ ]:
# Get data from my CV
path_my_cv = "/content/drive/MyDrive/CVs/CV_Data_Engineer_Caroline.pdf"
cv_md_text = pymupdf4llm.to_markdown(path_my_cv)
print(cv_md_text)

In [ ]:
%%time
# Save CV structured data to DB1 (for CVs)
doc_id = generate_file_doc_id(path_my_cv)
print('doc_id: ', doc_id)

cv_obj = extract_professional_structured_data(cv_md_text)
print('cv_obj: ', cv_obj)

newly_stored_profile = store_data_to_db(cv_obj, DB1_CURRICULUM_VITAE, doc_id=doc_id)
print('newly_stored_profile: ', newly_stored_profile)

# Check on whether DB2 (for Jobs) needs update for a given position or its variations
positions_to_check_on_updates = ", ". join(newly_stored_profile["json_data"]["main_position_name_variations"])
print('positions_to_check_on_updates: ', positions_to_check_on_updates)

must_update_position_in_db = True
now_timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

for job in DB2_JOBS:
    if job.get("json_data",{}).get("main_position_name").lower() in positions_to_check_on_updates.lower():
        creation_timestamp = datetime.strptime(job["creation_timestamp"], '%Y-%m-%d %H:%M:%S')
        if now_timestamp - creation_timestamp < timedelta(minutes=5):
            must_update_position_in_db = False
            break

if must_update_position_in_db:
    print("Calling Microsservice 2 to update Jobs Database (DB2_JOBS)")
else:
    print("No need to update Jobs Database (DB2_JOBS). Calling Microsservice 3 to find the best job matches for CV.")


## Complete Flow for sample CVs

In [ ]:
# Get data from random CVs sample

# Pick 5 random resume files paths
path_resumes_folder = '/content/drive/MyDrive/data'
path_categories = [f"{path_resumes_folder}/{category}" for category in os.listdir(path_resumes_folder)]

full_paths_resumes = []
for path_category in path_categories:
  all_category_resumes = os.listdir(path_category)
  full_paths_resumes += [f"{path_category}/{resume}" for resume in all_category_resumes]

paths_pdf_resumes_sample = random.sample(full_paths_resumes, 5)
resumes_md_texts = []
for path in paths_pdf_resumes_sample:
  resumes_md_texts.append({"file_path": path, "md_text": pymupdf4llm.to_markdown(path)})

KeyboardInterrupt: 

In [ ]:
print(paths_pdf_resumes_sample)


['/content/drive/MyDrive/data/APPAREL/29028935.pdf',
 '/content/drive/MyDrive/data/ARTS/27936502.pdf',
 '/content/drive/MyDrive/data/DESIGNER/13774329.pdf',
 '/content/drive/MyDrive/data/ACCOUNTANT/25749150.pdf',
 '/content/drive/MyDrive/data/ENGINEERING/17103000.pdf']

In [ ]:
%%time
from concurrent.futures import ThreadPoolExecutor

def process_single_resume(resume_data):
    file_path = resume_data["file_path"]
    md_text = resume_data["md_text"]

    doc_id = generate_file_doc_id(file_path)
    cv_obj = extract_professional_structured_data(md_text)
    return doc_id, cv_obj

processed_results = []
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_single_resume, resume) for resume in resumes_md_texts]
    for future in futures:
        doc_id, cv_obj = future.result()
        processed_results.append({"doc_id": doc_id, "cv_obj": cv_obj})

for result in processed_results:
    newly_stored_profile = store_data_to_db(result["cv_obj"], DB1_CURRICULUM_VITAE, doc_id=result["doc_id"])
    print(f'Newly stored profile: {newly_stored_profile["doc_id"]}')


Newly stored profile: 20251227214819e43554b8be6a4f9b6f3034314ecaa097a6f471de
Newly stored profile: 2025122721481902fb5ac1644bcf78e759565638d0cbde946b24c1
Newly stored profile: 20251227214819d0caa4f5d10c8fca65aa36ca62aea70c3cd10fe3
Newly stored profile: 202512272148194b66a4ccff8b4512ae93ae4dc081bf5c949cf43b
Newly stored profile: 20251227214819a4fbb25e8cbdf5c6397f0c50b59c60dfcc0faabc
CPU times: user 1.54 s, sys: 21.1 ms, total: 1.56 s
Wall time: 20.7 s


In [ ]:
# Check on whether DB2 (for Jobs) needs update for a given position or its variations
MAXIMUM_DATABASE_UPDATE_TIME_PERIOD_FOR_POSITION = 5

for cv in DB1_CURRICULUM_VITAE:

    now_timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    positions_to_check_on_updates = ", ". join(cv["json_data"]["main_position_name_variations"]).upper()
    goal_field = cv["json_data"]["main_position"][0]["field"].upper()
    print('positions_to_check_on_updates: ', positions_to_check_on_updates)
    print('goal_field: ', goal_field)

    must_update_position_in_db = True
    is_full_position_found_in_db = False
    is_position_update_expired = False

    for job in DB2_JOBS:
        job_main_position       = job.get("json_data",{}).get("main_position", {}).get("name").upper()
        job_main_position_field = job.get("json_data",{}).get("main_position", {}).get("field").upper()
        cv_position_matches_job = job_main_position.upper() in positions_to_check_on_updates
        cv_field_matches_job    = job_main_position_field.upper() in goal_field

        if cv_position_matches_job and cv_field_matches_job:
            is_full_position_found_in_db = True
            creation_timestamp = datetime.strptime(job["creation_timestamp"], '%Y-%m-%d %H:%M:%S')
            time_spent_since_database_update = now_timestamp - creation_timestamp
            if time_spent_since_database_update < timedelta(minutes=MAXIMUM_DATABASE_UPDATE_TIME_PERIOD_FOR_POSITION):
                is_position_update_expired = True
                break
            else:
                must_update_position_in_db = False
                break

    if must_update_position_in_db:
        print("Calling Microsservice 2 to update Jobs Database (DB2_JOBS)", sep=". ")
        if not is_full_position_found_in_db:
            print("The position couldn't be found in the database yet.")
        elif is_position_update_expired:
            print("The position was found, but time spent since the last update is higher than the default standard period for database update.")
    else:
        print("No need to update Jobs Database (DB2_JOBS). Calling Microsservice 3 to find the best job matches for CV.")

    print('\n')

positions_to_check_on_updates:  RESERVATIONS AGENT, FRONT DESK AGENT, GUEST SERVICES AGENT, PBX OPERATOR
goal_field:  HOSPITALITY
Calling Microsservice 2 to update Jobs Database (DB2_JOBS)
The position couldn't be found in the database yet.


positions_to_check_on_updates:  PACKAGING BUYER, RESEARCH AND DEVELOPMENT LEAD, OPERATIONS MANAGER, PURCHASING MANAGER
goal_field:  OPERATIONS/PURCHASING/R&D
Calling Microsservice 2 to update Jobs Database (DB2_JOBS)
The position couldn't be found in the database yet.


positions_to_check_on_updates:  MECHANICAL DESIGNER, R&D DESIGNER, FOREMAN/DESIGNER, DESIGNER/PROJECT MANAGER
goal_field:  MECHANICAL DESIGN
Calling Microsservice 2 to update Jobs Database (DB2_JOBS)
The position couldn't be found in the database yet.


positions_to_check_on_updates:  ACCOUNTANT, SENIOR ACCOUNTANT
goal_field:  ACCOUNTING
Calling Microsservice 2 to update Jobs Database (DB2_JOBS)
The position couldn't be found in the database yet.


positions_to_check_on_updates:  M

# Microsservice 2
